# P21 — Pearson-Correlation Graph Construction (Arm B vs Arm A)

**The one lever:** replace the fixed full topology with a **per-window Pearson-correlation
adjacency** (Yang et al. 2024, GDA-inspired). Backbone stays GraphSAGE; ConvNet branch,
fusion, splits and selection are frozen. Both arms are **DG protocol** (source-only).

**Reuse — nothing baseline retrains:** Arm A = the P18b `base_s42/s1/s2` cells
(76.87 ± 1.07%), copied in together with the per-seed ConvNet probs (frozen branch).

**Compute:** Stage A ≈ 4 GNN-only arms (~25 min each on a T4). Stage B ≈ 2 more
(seeds 1, 2 of the winner). No ConvNet training anywhere.

**Gate:** seed-mean fused Δ ≥ +1.0pp vs base AND no fold < −2pp, else REVERT.

> **Final_PROJECT note (updated 2026-09-26).** All paths now come from
> `Final_PROJECT/project_paths.py` via the first code cell below — the only thing a new user may
> need to edit is the DAGHAR data path in that file (and `FINAL_PROJECT_ROOT` below, only if the
> folder isn't found automatically). **Saved cell outputs are from the original thesis run**, when
> this code lived at `/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph`; they are kept as the record of that run.
> All training scripts are **resumable**: re-running this notebook in place skips any fold whose
> result files already exist, so it will not overwrite the thesis results.

## 1. Mount & paths

In [1]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("p21")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH
# Frozen ConvNet probabilities (seeds 42/1/2) from the P18b source-only run
P18B_SWEEP   = PP.CONVNET_FROZEN_SWEEP
assert os.path.isdir(P18B_SWEEP), P18B_SWEEP

Mounted at /content/drive
Final_PROJECT paths
  PROJECT_ROOT : /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT
  DATA_PATH    : /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  component    : p21 -> /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT/01_Contribution1_Generalization_Model/P21_PearsonGraph
  cwd          : /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT/01_Contribution1_Generalization_Model/P21_PearsonGraph


## 2. Setup

In [2]:
import setup_colab
setup_colab.install_deps(); assert setup_colab.verify(), "fix DATA_PATH"

Installing PyTorch Geometric ...
Done.

Verifying imports ...
  OK torch 2.11.0+cu128  CUDA=True
     GPU Tesla T4
  OK torch_geometric 2.8.0.post1

Checking data at: /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  OK KuHar              3 CSVs
  OK MotionSense        3 CSVs
  OK RealWorld-Thigh    3 CSVs
  OK RealWorld-Waist    3 CSVs
  OK UCI                3 CSVs
  OK WISDM              3 CSVs


## 3. Smoke test (~3 min)
One fold, 2 epochs, Pearson edges — confirms the per-window graph construction builds,
logs edge statistics, trains, and dumps an aligned npz.

In [3]:
# Smoke test writes to a fresh throwaway folder, so it always really trains (the saved
# results/smoke from the original run would otherwise be skipped as "already done").
import shutil; shutil.rmtree("results/_smoke_test", ignore_errors=True)
!python -m p21.gnn_predict --data_path "$DATA_PATH" --targets KuHar --epochs 2 --log_every 1 \
    --edge_mode pearson --select topk --topk 9 --alpha 0.5 --seed 42 \
    --out_dir results/_smoke_test
import numpy as np; z = np.load("results/_smoke_test/gnn_KuHar.npz")
print("smoke OK:", z["probs"].shape, z["labels"].shape)
shutil.rmtree("results/_smoke_test"); print("smoke output removed")

[GNN] P21 | edges=pearson select=topk topk=9 tau=0.3 alpha=0.5 | seed=42 | P9 recipe, source-only (DG) | targets=['KuHar'] | out=results/_smoke_test
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (4, 5):100%, (3, 5):100%, (1, 4):100%, (2, 5):100%
    [pearson test] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (1, 2):100%, (4, 5):100%, (0, 1):100%, (1, 4):100%
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 18 edges, 90 features each)...
    bu

## 4. STAGE A — config sweep at seed 42 (~1.5–2 h)
Four Pearson configs, **selected on mean source-val accuracy** (never target-test):
`topk9_a1.0`, `topk6_a1.0`, `topk12_a1.0`, `topk9_a0.5` (blend with physical prior).
Resumable — finished folds are skipped on re-run.

In [ ]:
!python -m p21.run_sweep --data_path "$DATA_PATH" --seed 42 --epochs 100 --log_every 10


>>> [topk9_a1.0] p21.gnn_predict --data_path /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view --targets all --epochs 100 --log_every 10 --seed 42 --edge_mode pearson --out_dir /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep_cfg/topk9_a1.0 --select topk --topk 9 --alpha 1.0
[GNN] P21 | edges=pearson select=topk topk=9 tau=0.3 alpha=1.0 | seed=42 | P9 recipe, source-only (DG) | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep_cfg/topk9_a1.0
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=1.0
    [pearson train] edges/window: mean 9.0  min 9  max 9

## 5. STAGE B — seed-average the winner (~1 h)
Set `WINNER` from Stage A. Copies the P18b base cells + per-seed ConvNet probs
(no retraining), reuses the Stage A run as seed 42, trains seeds 1 and 2.

> *Historical cell:* this first Stage-B run used the provisional winner `topk9_a1.0`. The Stage-A source-val winner actually adopted is **`topk9_a0.5`** — see "Stage B (Real/Updated)" below.

In [ ]:
WINNER = "topk9_a1.0"   # <-- set from the Stage A report
!python -m p21.run_seeds --data_path "$DATA_PATH" --config {WINNER} --seeds 42,1,2 \
    --epochs 100 --log_every 10 --p18b_sweep "{P18B_SWEEP}"

[reuse] base_s42: copied 12 npz from P18b
[reuse] base_s1: copied 12 npz from P18b
[reuse] base_s2: copied 12 npz from P18b
[reuse] pearson_topk9_a1.0_s42: copied 6 gnn npz from Stage A
[skip] pearson_topk9_a1.0_s42: gnn complete
[reuse] pearson_topk9_a1.0_s42: copied 6 convnet npz (frozen branch, seed 42)

>>> [pearson_topk9_a1.0 s1] p21.gnn_predict --data_path /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view --targets all --epochs 100 --log_every 10 --seed 1 --edge_mode pearson --out_dir /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep/pearson_topk9_a1.0_s1 --select topk --topk 9 --alpha 1.0
[GNN] P21 | edges=pearson select=topk topk=9 tau=0.3 alpha=1.0 | seed=1 | P9 recipe, source-only (DG) | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep/pearson_topk9_a1.0_s1
[LODO] targ

## 6. Seed-averaged report + gate

In [ ]:
!python -m p21.run_report


  P21 — SEED-AVERAGED: Pearson graph construction vs fixed topology
  (both arms DG protocol — source-only, frozen ConvNet branch & fusion)

  condition              seeds  fused mean±std  GNN-only   Δ vs base  gate
  ------------------------------------------------------------------------
  base                       3     76.87±1.07%    72.32%  (baseline)
  pearson_topk9_a1.0         3     76.98±0.41%    71.42%     +0.11pp  revert

  Per-fold seed-means — base vs pearson_topk9_a1.0 (fused | GNN-only):
  fold                 base pearson_topk9_a1.0     Δpp    base-GNN  pears-GNN
  --------------------------------------------------------------------
  KuHar              65.05%     65.74%  +0.69      64.81%     59.72%
  MotionSense        85.03%     85.59%  +0.56      78.37%     79.32%
  RealWorld-Thigh    75.68%     73.59%  -2.09      70.17%     67.56%
  RealWorld-Waist    75.13%     77.60%  +2.47      70.36%     73.66%
  UCI                86.18%     84.35%  -1.84      78.50%     75.

---
### Reading it
- **KEEP** only if seed-mean fused Δ ≥ **+1.0pp** vs base with **no fold < −2pp**.
- The `GNN-only` column shows whether Pearson edges helped the GNN branch itself —
  even a sub-gate fused result is informative for P22 if GNN-only moved.
- **KEEP →** P22 runs Residual ChebNet on **Pearson** edges (Arm D path, and ChebNet
  can consume the |ρ| values as true edge *weights*, which SAGE cannot).
- **REVERT →** P22 runs Residual ChebNet on **fixed** edges (Arm C), and we record that
  on a 6-node single-position graph, data-driven topology adds nothing for SAGE.

### Optional extras (cheap, only if Stage A is ambiguous)
```
!python -m p21.run_sweep --data_path "$DATA_PATH" --configs topk6_a1.0,topk12_a1.0
```

## Stage B (Real/Updated)

In [ ]:
WINNER = "topk9_a0.5"
!python -m p21.run_seeds --data_path "$DATA_PATH" --config {WINNER} --seeds 42,1,2 --epochs 100 --log_every 10 --p18b_sweep "{P18B_SWEEP}"
!python -m p21.run_report

[reuse] base_s42: already present
[reuse] base_s1: already present
[reuse] base_s2: already present
[reuse] pearson_topk9_a0.5_s42: copied 6 gnn npz from Stage A
[skip] pearson_topk9_a0.5_s42: gnn complete
[reuse] pearson_topk9_a0.5_s42: copied 6 convnet npz (frozen branch, seed 42)

>>> [pearson_topk9_a0.5 s1] p21.gnn_predict --data_path /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view --targets all --epochs 100 --log_every 10 --seed 1 --edge_mode pearson --out_dir /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep/pearson_topk9_a0.5_s1 --select topk --topk 9 --alpha 0.5
[GNN] P21 | edges=pearson select=topk topk=9 tau=0.3 alpha=0.5 | seed=1 | P9 recipe, source-only (DG) | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep/pearson_topk9_a0.5_s1
[LODO] target = KuHar
[LODO] source

---
ADDITIONALS for 5-seeds report
---

## Train base (fixed-edge) GNN for seeds 3 and 5, copy to P18b sweep

In [ ]:
import os, shutil, glob

# Arm-A (fixed-edge) GNN for seeds 3 and 5, written straight into THIS folder's
# results/sweep/base_s{seed}. (In the original run these were staged through the P18b
# sweep folder; writing them here gives run_seeds.py the same complete cells.)
for seed in [3, 5]:
    out_dir = f"results/sweep/base_s{seed}_tmp"
    # Train base (fixed edges = Arm A control) for this seed — resumable
    !python -m p21.gnn_predict --data_path "$DATA_PATH" --targets all \
        --epochs 100 --log_every 10 --seed {seed} \
        --edge_mode fixed \
        --out_dir {out_dir}

    dst = os.path.join("results", "sweep", f"base_s{seed}")
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f"{out_dir}/gnn_*.npz"):
        shutil.copy2(f, dst)

    # ConvNet branch is frozen: seeds 3 and 5 reuse the seed-42 ConvNet probabilities
    # as a stand-in (exactly as in the thesis run; files are byte-identical).
    for f in glob.glob(os.path.join(P18B_SWEEP, "base_s42", "convnet_*.npz")):
        if not os.path.exists(os.path.join(dst, os.path.basename(f))):
            shutil.copy2(f, dst)

    print(f"✓ base_s{seed} ready at {dst}")

[GNN] P21 | edges=fixed(full) | seed=3 | P9 recipe, source-only (DG) | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=results/sweep/base_s3_tmp
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 30 edges, 90 features each)...
    built 5000/35396 graphs
    built 10000/35396 graphs
    built 15000/35396 graphs
    built 20000/35396 graphs
    built 25000/35396 graphs
    built 30000/35396 graphs
    built 35000/35396 graphs
    done: 35396 graphs built
  [gnn:KuHar] source-train=35396  source-val=5412  target-test=144  edge_mode=fixed
    [gnn:KuHar] epoch   1 | act_loss 0.7659 | dom_loss 1.4248 | lambda 0.050 | tr_acc  68.1% | val_loss 0.6525 | val_acc  75.4%  *
    [gnn:

## Now run Pearson seeds 3 and 5

In [ ]:
WINNER = "topk9_a0.5"
!python -m p21.run_seeds --data_path "$DATA_PATH" --config {WINNER} \
    --seeds 42,1,2,3,5 --epochs 100 --log_every 10 --p18b_sweep "{P18B_SWEEP}"
!python -m p21.run_report

[reuse] base_s42: already present
[reuse] base_s1: already present
[reuse] base_s2: already present
[reuse] base_s3: copied 12 npz from P18b
[reuse] base_s5: copied 12 npz from P18b
[skip] pearson_topk9_a0.5_s42: gnn complete
[skip] pearson_topk9_a0.5_s1: gnn complete
[skip] pearson_topk9_a0.5_s2: gnn complete

>>> [pearson_topk9_a0.5 s3] p21.gnn_predict --data_path /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view --targets all --epochs 100 --log_every 10 --seed 3 --edge_mode pearson --out_dir /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep/pearson_topk9_a0.5_s3 --select topk --topk 9 --alpha 0.5
[GNN] P21 | edges=pearson select=topk topk=9 tau=0.3 alpha=0.5 | seed=3 | P9 recipe, source-only (DG) | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P21_PearsonGraph/results/sweep/pearson_topk9_a0.5_s3
[LODO] 

## Regenerate the seed report with all 5 seeds

In [ ]:
!python -m p21.run_report


  P21 — SEED-AVERAGED: Pearson graph construction vs fixed topology
  (both arms DG protocol — source-only, frozen ConvNet branch & fusion)

  condition              seeds  fused mean±std  GNN-only   Δ vs base  gate
  ------------------------------------------------------------------------
  base                       5     76.68±1.36%    72.05%  (baseline)
  pearson_topk9_a0.5         5     77.96±0.21%    74.41%     +1.28pp  KEEP
  pearson_topk9_a1.0         3     76.98±0.41%    71.42%     +0.30pp  revert

  Per-fold seed-means — base vs pearson_topk9_a0.5 (fused | GNN-only):
  fold                 base pearson_topk9_a0.5     Δpp    base-GNN  pears-GNN
  --------------------------------------------------------------------
  KuHar              66.11%     67.50%  +1.39      62.08%     64.58%
  MotionSense        84.93%     87.06%  +2.13      79.00%     82.18%
  RealWorld-Thigh    74.85%     74.93%  +0.08      70.20%     71.35%
  RealWorld-Waist    74.75%     76.49%  +1.74      70.43%  

## Compute precision / recall / macro-F1 from stored .npz files (runs locally, no retraining)

In [ ]:
import numpy as np, json, glob, os, re
from sklearn.metrics import precision_recall_fscore_support, classification_report

SWEEP = "results/sweep"
WINNER = "pearson_topk9_a0.5"
TARGETS = ["KuHar", "MotionSense", "RealWorld-Thigh", "RealWorld-Waist", "UCI", "WISDM"]
WEIGHTS = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]
FUSED_W = 0.6  # confirmed best_w across all P21 seeds

# --- collect per-seed, per-fold predictions ---
seed_dirs = sorted(glob.glob(os.path.join(SWEEP, f"{WINNER}_s*")))
seed_ids  = [int(re.search(r"_s(\d+)$", d).group(1)) for d in seed_dirs]
print(f"Seeds found: {seed_ids}")

# per-seed metrics
all_seed_metrics = {}
for d, sid in zip(seed_dirs, seed_ids):
    fold_metrics = {}
    all_preds, all_labels = [], []
    for t in TARGETS:
        G = np.load(os.path.join(d, f"gnn_{t}.npz"))
        C = np.load(os.path.join(d, f"convnet_{t}.npz"))
        y = G["labels"]
        fused = FUSED_W * G["probs"] + (1 - FUSED_W) * C["probs"]
        preds = fused.argmax(1)
        classes_present = np.unique(y)
        p, r, f, _ = precision_recall_fscore_support(
            y, preds, labels=classes_present, average="macro", zero_division=0)
        fold_metrics[t] = {"precision": round(p*100,2), "recall": round(r*100,2), "f1": round(f*100,2),
                           "accuracy": round((preds==y).mean()*100,2), "n": int(len(y))}
        all_preds.append(preds); all_labels.append(y)
    # overall across all folds
    yall = np.concatenate(all_labels); pall = np.concatenate(all_preds)
    classes_all = np.unique(yall)
    p, r, f, _ = precision_recall_fscore_support(yall, pall, labels=classes_all, average="macro", zero_division=0)
    fold_metrics["OVERALL"] = {"precision": round(p*100,2), "recall": round(r*100,2), "f1": round(f*100,2),
                               "accuracy": round((pall==yall).mean()*100,2)}
    all_seed_metrics[sid] = fold_metrics

# --- seed-averaged summary ---
print(f"\n{'Fold':18s} {'Acc':>7s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s}  (macro, mean over {len(seed_ids)} seeds)")
print("-"*60)
summary = {}
for t in TARGETS + ["OVERALL"]:
    accs  = [all_seed_metrics[s][t]["accuracy"]  for s in seed_ids]
    precs = [all_seed_metrics[s][t]["precision"] for s in seed_ids]
    recs  = [all_seed_metrics[s][t]["recall"]    for s in seed_ids]
    f1s   = [all_seed_metrics[s][t]["f1"]        for s in seed_ids]
    summary[t] = {"acc_mean": round(np.mean(accs),2), "acc_std": round(np.std(accs),2),
                  "prec_mean": round(np.mean(precs),2), "rec_mean": round(np.mean(recs),2),
                  "f1_mean": round(np.mean(f1s),2), "f1_std": round(np.std(f1s),2)}
    row = summary[t]
    print(f"  {t:16s} {row['acc_mean']:6.2f}% {row['prec_mean']:9.2f}% "
          f"{row['rec_mean']:7.2f}% {row['f1_mean']:7.2f}%±{row['f1_std']:.2f}")

# save
out = {"winner": WINNER, "seeds": seed_ids, "fused_w": FUSED_W,
       "per_seed": all_seed_metrics, "seed_avg_summary": summary}
with open("results/p21_full_metrics.json", "w") as f:
    json.dump(out, f, indent=2)
print("\nSaved: results/p21_full_metrics.json")

Seeds found: [1, 2, 3, 42, 5]

Fold                   Acc  Precision   Recall       F1  (macro, mean over 5 seeds)
------------------------------------------------------------
  KuHar             67.50%     77.13%   67.50%   63.10%±3.10
  MotionSense       87.06%     88.06%   87.06%   86.91%±2.48
  RealWorld-Thigh   74.94%     79.89%   74.94%   74.21%±0.83
  RealWorld-Waist   76.49%     79.06%   76.49%   77.04%±1.20
  UCI               86.38%     88.08%   86.38%   86.03%±0.74
  WISDM             75.38%     82.75%   75.38%   78.62%±0.98
  OVERALL           77.43%     79.01%   77.63%   78.06%±0.18

Saved: results/p21_full_metrics.json
